# MoodTune — Phase 1: Dataset Exploration

This notebook performs read-only inspection of the original dataset in `data/raw/`. It does not clean, modify, or write the raw source data. Run it only after placing the Kaggle dataset in that folder.

In [ ]:
from pathlib import Path
import pandas as pd

project_roots = [Path.cwd(), *Path.cwd().parents]
RAW_DATA_DIR = next(
    (root / 'data' / 'raw' for root in project_roots if (root / 'data' / 'raw').is_dir()),
    None,
)
if RAW_DATA_DIR is None:
    raise FileNotFoundError('Run this notebook from the MoodTune project or a child directory.')
SUPPORTED_SUFFIXES = {'.csv'}
dataset_files = [
    path for path in RAW_DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
]

if len(dataset_files) != 1:
    raise FileNotFoundError(
        'Place exactly one CSV dataset in data/raw/ before running this notebook. '
        f'Found: {[path.name for path in dataset_files]}'
    )

DATASET_PATH = dataset_files[0]
print(f'Dataset: {DATASET_PATH.name}')
print(f'Format: {DATASET_PATH.suffix.lower()}')
print(f'Size: {DATASET_PATH.stat().st_size:,} bytes')
df = pd.read_csv(DATASET_PATH)
df.shape

In [ ]:
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns):,}')
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
display(df.dtypes.rename('dtype').to_frame())
print('\nExample records:')
display(df.head())

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100).round(2),
}).sort_values(['missing_count', 'missing_percentage'], ascending=False)
display(missing)
print(f'Exact duplicate rows: {df.duplicated().sum():,}')

In [ ]:
identifier_candidates = [column for column in df.columns if any(
    token in column.lower() for token in ('track_id', 'id', 'uri')
)]
print('Potential identifier columns:', identifier_candidates)
for column in identifier_candidates:
    print(f'{column}: {df[column].nunique(dropna=True):,} unique non-null values; ' f'{df[column].duplicated().sum():,} repeated values')

categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
print('\nCategorical columns:', categorical_columns)
for column in categorical_columns:
    print(f'\n{column}: {df[column].nunique(dropna=True):,} unique non-null values')
    display(df[column].value_counts(dropna=False).head(10).rename('count').to_frame())

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()
print('Numerical columns:', numeric_columns)
display(df[numeric_columns].describe().T)

audio_feature_keywords = (
    'valence', 'energy', 'danceability', 'acousticness', 'instrumentalness',
    'speechiness', 'loudness', 'tempo', 'liveness', 'popularity', 'duration'
)
audio_feature_columns = [
    column for column in numeric_columns
    if any(keyword in column.lower() for keyword in audio_feature_keywords)
]
print('\nPotential mood/recommendation features:', audio_feature_columns)
display(df[audio_feature_columns].agg(['min', 'max', 'mean', 'median']).T)

## Phase 1 interpretation checklist

After running, record the dataset-specific output in `docs/dataset_exploration.md`: file details, schema, missingness, duplicate patterns, genre/playlist values, numerical ranges, suspicious values, and a column-usage table. Do not start cleaning in this notebook.